In [1]:
# Imports
import sys, time, math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path.cwd().parent))

from common.utils import set_seed, get_device, count_parameters
from common.datasets import (FASHION_MNIST_CLASSES, FASHION_MNIST_SHAPE, 
                             load_fashion_mnist, load_or_create_split,
                             pixel_stats, to_tensor_dataset,
                             make_train_loader, make_eval_loaders)
from common.training import fit

# Config
CONFIG = {
    "dataset": "FashionMNIST",
    "data_dir": Path("../../fashion-mnist"),
    "seed": 42,
    "val_ratio": 0.1,
    "batch_size": 128,
    "epochs": 50,
    "lr": 1e-3,          
    "weight_decay": 0.0,
    "patience": 5,
}
SPLIT_PATH = Path(f"../E1/split_indices_seed{CONFIG['seed']}.npz")   # cùng split với E1

set_seed(CONFIG["seed"])
DEVICE = get_device()


## MSA

Input $X \in \mathbb{R}^{B \times N \times D}$, gồm $n$ token, mỗi token có $D$ chiều, chia thành $h$ head với $d = D/h$, $B$ là số lượng sample.

Giả sử ta đem các giá trị vào một Tensor, với B là số lượng sample, N là số lượng token, thì:

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V$$
$$\text{head}_i = \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{d}}\right) V_i$$
$$\text{MSA}(X) = \text{Concat}(\text{head}_1,\dots,\text{head}_h)\, W_O$$

Cụ thể, $Q,K,V \in \mathbb{R}^{B \times N \times d} \Rightarrow QK^\top \in \mathbb{R}^{B \times N \times N}$ hoặc $Q,K,V \in \mathbb{R}^{B \times h \times N \times d} \Rightarrow QK^\top \in \mathbb{R}^{B \times h \times N \times N}$ 

$\text{head}_i \in \mathbb{R}^{B \times N \times N}$

In [2]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
      super().__init__()
      self.embed_dim = embed_dim                # D
      self.num_heads = num_heads                # h
      self.head_dim = embed_dim // num_heads    # D/h
      self.scale = 1 / math.sqrt(self.head_dim)
      self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=bias) # W_q, W_k, W_v combined
      self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias) # Output projection
      self.attn_dropout = nn.Dropout(attn_dropout)

    def forward(self, x, return_attn: bool = False):
      B, N, D = x.shape

      # 1. Tính Q, K, V
      qkv = self.qkv(x)  # (B, N, 3*D)
      qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)  # (3, B, h, N, D/h)
      q, k, v = qkv[0], qkv[1], qkv[2]  # Each is (B, h, N, D/h)

      # 2. Tính attention scores bằng einsum
      attn_scores = torch.einsum('bhqd,bhkd->bhqk', q, k) * self.scale  # (B, h, N, N)
      attn_probs = F.softmax(attn_scores, dim=-1)
      attn_probs = self.attn_dropout(attn_probs)

      # 3. Tính weighted sum của V
      attn_output = torch.einsum('bhqk,bhkd->bhqd', attn_probs, v)  # (B, h, N, D/h)
      attn_output = attn_output.permute(0, 2, 1, 3).reshape(B, N, D)  # (B, N, D)

      # 4. Áp dụng output projection
      output = self.out_proj(attn_output)

      if return_attn:
          return output, attn_probs
      return output


In [3]:
class MSAPyTorch(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            dropout=attn_dropout, 
            bias=bias, 
            batch_first=True # Để đầu vào có dạng (B, N, D) thay vì (N, B, D)
        )

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        attn_output, attn_weights = self.mha(x, x, x, need_weights=return_attn, average_attn_weights=False)
        if return_attn:
            return attn_output, attn_weights
        return attn_output

In [4]:
ATTENTIONS = {
  "scratch": MultiHeadSelfAttention,
  "torch": MSAPyTorch,
}

# Ánh xạ hậu tố key: scratch -> torch
_MSA_KEY_MAP = {
  "qkv.weight": "mha.in_proj_weight",
  "qkv.bias": "mha.in_proj_bias",
  "out_proj.weight": "mha.out_proj.weight",
  "out_proj.bias": "mha.out_proj.bias",
}

def scratch_to_torch_state_dict(state_dict: dict) -> dict:
    torch_state_dict = {}
    for key, value in state_dict.items():
        # Nếu key kết thúc bằng một hậu tố trong _MSA_KEY_MAP, ánh xạ sang key tương ứng của torch
        for suffix, torch_key in _MSA_KEY_MAP.items():
            if key.endswith(suffix):
                torch_state_dict[key[:-len(suffix)] + torch_key] = value
                break
        else:
            torch_state_dict[key] = value
    return torch_state_dict

In [5]:
# Kiểm tra hai mô hình Attention

@torch.no_grad()
def check_scratch_and_torch(B=4, N=20, D=64, h=4, atol=1e-5):
    set_seed(0)
    scratch = MultiHeadSelfAttention(embed_dim=D, num_heads=h).eval()
    torch_msa = MSAPyTorch(embed_dim=D, num_heads=h).eval()
    torch_msa.load_state_dict(scratch_to_torch_state_dict(scratch.state_dict()))

    # Tạo đầu vào ngẫu nhiên
    x = torch.randn(B, N, D)

    # Kiểm tra đầu ra của scratch và torch có gần nhau không
    out_scratch, attn_scratch = scratch(x, return_attn=True)
    out_torch, attn_torch = torch_msa(x, return_attn=True)
    assert torch.allclose(out_scratch, out_torch, atol=atol) and torch.allclose(attn_scratch, attn_torch, atol=atol), "Outputs are not close!"

    # Tính số tham số
    n_s, n_t = count_parameters(scratch), count_parameters(torch_msa)
    print(f"Number of parameters - Scratch: {n_s}, Torch: {n_t}")

    # Kiểm tra số tham số phải bằng nhau và bằng 4D^2 + 4D
    assert n_s == n_t == 4 * D**2 + 4 * D, "Number of parameters is not correct!"

check_scratch_and_torch()

Number of parameters - Scratch: 16640, Torch: 16640


In [6]:
# Đo tốc độ chạy của hai mô hình Attention
def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

def bench_attention(cls, N, D=64, h=4, B=128, iters=50, warmup=10, device=DEVICE):
    set_seed(0)
    module = cls(embed_dim=D, num_heads=h).to(device).train()
    x = torch.randn(B, N, D, device=device, requires_grad=True)
    for i in range(warmup + iters):
        if (i == warmup):
            sync(device)
            t0 = time.perf_counter()
        module(x).sum().backward()
    sync(device)
    t1 = time.perf_counter()
    return (t1 - t0) / iters * 1e3  # Convert to milliseconds

# N = số token với ảnh 28×28:
#   16  = patch size 7 → lưới 4×4
#   28  = mỗi hàng là 1 token
#   49  = patch size 4 → lưới 7×7
#   196 = patch size 2 → lưới 14×14
for N in (16, 28, 49, 196):
    times = {
        name: bench_attention(cls, N) for name, cls in ATTENTIONS.items()
    }
    print(f"N={N:>4}: " + "  ".join(f"{k}={v:6.2f} ms" for k, v in times.items()))

N=  16: scratch=  1.95 ms  torch=  1.90 ms
N=  28: scratch=  2.58 ms  torch=  2.02 ms
N=  49: scratch=  4.63 ms  torch=  4.70 ms
N= 196: scratch= 40.96 ms  torch= 40.34 ms


In [ ]:
from ..E1.classifiers import SoftmaxClassifier, MLPClassifier

class TransformerClassifier(nn.Module):
    def __init__(self, tokenizer, attention="scratch", embed_dim=64, depth=2, num_heads=4, 
                 mlp_ratio=2, dropout=0.1, pool="cls", head="softmax", use_pos=True):
        super().__init__()
        self.tokenizer = tokenizer
        self.pool = pool
        num_tokens = tokenizer.num_tokens
        if pool == "cls":
            num_tokens += 1

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim)) if pool == "cls" else None
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens, embed_dim)) if use_pos else None
        self.pos_drop = nn.Dropout(p=dropout) if use_pos else None
        self.blocks = nn.ModuleList([
            nn.Sequential(
                ATTENTIONS[attention](embed_dim=embed_dim, num_heads=num_heads),
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim * mlp_ratio),
                nn.GELU(),
                nn.Linear(embed_dim * mlp_ratio, embed_dim),
                nn.Dropout(p=dropout)
            )
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = SoftmaxClassifier(input_dim=embed_dim, num_classes=tokenizer.num_tokens) if head == "softmax" else MLPClassifier(input_dim=embed_dim, hidden_dim=embed_dim * mlp_ratio, num_classes=tokenizer.num_tokens)
        
        if self.pos_embed is not None:
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.cls_token is not None:
            nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        tokens = self.tokenizer(x)
        if self.cls_token is not None:
            cls_tokens = self.cls_token.expand(tokens.size(0), -1, -1)
            tokens = torch.cat((cls_tokens, tokens), dim=1)
        if self.pos_embed is not None:
            tokens = tokens + self.pos_embed
            tokens = self.pos_drop(tokens)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)
        if self.pool == "cls":
            tokens = tokens[:, 0]
        else:
            tokens = tokens.mean(dim=1)
        return self.head(tokens)

    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.softmax(self(x), dim=-1)

In [ ]:
# Patching

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=7, embed_dim=64, img_size=28, in_channels=1):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.img_size = img_size
        self.in_channels = in_channels
        self.num_tokens = (img_size // patch_size) ** 2
        self.proj = nn.Linear(patch_size * patch_size * in_channels, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size

        # [B, C, H/p, p, W/p, p] -> [B, H/p, W/p, C, p, p]
        x = x.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 1, 3, 5)
        x = x.reshape(B, -1, C * p * p)
        return self.proj(x)

In [ ]:
# Test 1 shape và bảng tham số

set_seed(0)
model = TransformerClassifier(tokenizer=PatchEmbedding(7, 64))
x = torch.randn(8, 1, 28, 28)

for name, part in [("tokenizer", model.tokenizer), ("1 block", model.blocks[0]),
                   ("  ├ attn", model.blocks[0].attn), ("  └ ffn", model.blocks[0].ffn),
                   ("head", model.head), ("TOTAL", model)]:
    print(f"{name:<10} {count_parameters(part):>8,}")

In [ ]:
# Test 2 — scratch và torch cho cùng logits khi dùng chung trọng số

@torch.no_grad()
def check_classifier_equivalence(atol=1e-5):
    set_seed(0)
    m_scratch = TransformerClassifier(PatchEmbedding(7, 64), attention="scratch").eval()
    m_torch = TransformerClassifier(PatchEmbedding(7, 64), attention="torch").eval()
    m_torch.load_state_dict(scratch_to_torch_state_dict(m_scratch.state_dict()))   # strict=True
    x = torch.randn(8, 1, 28, 28)
    diff = (m_scratch(x) - m_torch(x)).abs().max().item()
    print(f"max |Δlogits| = {diff:.2e}")
    assert diff < atol

check_classifier_equivalence()          # tham chiếu: ~6.9e-07